In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


df = pd.read_csv('student-mat.csv', sep=';') 

print(f"Dataset successfully loaded! Total student records: {len(df)}")

# ==========================================
# 2. DEFINE TARGET & DROP LEAKAGE FEATURES
# ==========================================
# 10 out of 20 is the standard passing threshold
df['pass'] = (df['G3'] >= 10).astype(int)

# Drop G1, G2, and G3 to keep predictions strictly based on early-year metrics
features_to_drop = ['G1', 'G2', 'G3', 'pass']
X = df.drop(columns=features_to_drop)
y = df['pass']



# ==========================================
# 3. PREPROCESSING & ENCODING
# ==========================================
# Convert all binary string objects ("yes"/"no", "GP"/"MS") and nominal features into dummy variables
# drop_first=True prevents the multicollinearity dummy variable trap!
X = pd.get_dummies(X, drop_first=True)

# Ensure all boolean dummy indicators are numeric 1s and 0s
X = X.astype(float)

# ==========================================
# 4. TRAIN-TEST SPLIT
# ==========================================
# Stratify ensures the pass/fail proportions remain identical in both splits
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ==========================================
# 5. FEATURE SCALING (MANDATORY FOR SVM)
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 6. HYPERPARAMETER TUNING & TRAINING
# ==========================================
# probability=True allows calculation of prediction probabilities for AUC-ROC metric profiles
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_svm = grid_search.best_estimator_
print(f"Optimal Hyperparameters: {grid_search.best_params_}\n")

# ==========================================
# 7. MODEL EVALUATION
# ==========================================
y_pred = best_svm.predict(X_test_scaled)
y_prob = best_svm.predict_proba(X_test_scaled)[:, 1]

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Performance ---")
print(classification_report(y_test, y_pred))

try:
    print(f"AUC-ROC Score: {roc_auc_score(y_test, y_prob):.4f}")
except ValueError:
    print("AUC-ROC score skipped (requires representation of both classes in sample array split)")


Dataset successfully loaded! Total student records: 395
Optimal Hyperparameters: {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}

--- Confusion Matrix ---
[[ 3 23]
 [ 4 49]]

--- Classification Performance ---
              precision    recall  f1-score   support

           0       0.43      0.12      0.18        26
           1       0.68      0.92      0.78        53

    accuracy                           0.66        79
   macro avg       0.55      0.52      0.48        79
weighted avg       0.60      0.66      0.59        79

AUC-ROC Score: 0.6009
